# Tutorial and Exercise: Attention and Visual Transfomer

In this exercise, we want to look at image classification again, but this time use a visual transformer for the cifar-10 data set that we will set up ourselves using the transformer classes in torch.

We have to install some additional packages (will be changed to be included in the base image :-) ).

We will also first play around with the attention calculation in torch.

In [ ]:
!pip install torchinfo torcheval

In [ ]:
import cv2
import os
import sys
import numpy as np
import math

import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torcheval.metrics import MulticlassAccuracy
import torchvision


sys.path.append('.')

# for displaying images in jupyter
import matplotlib as mpl
from matplotlib import pyplot as plt

from pathlib import Path

# not the correct directory, but necessary for the last exercise :-)

data_dir = Path('/exchange/cvai/images')

## Attention and transformer mechanisms in torch

### Vectors 
We can construct random vectors like this, in general we will only be interested in the size of the output. Here the vector v simulates a sequence of 10 values with an embedding size of 4.

In [ ]:
v = torch.rand([10,4])
print(v)
print(v.shape)

### Calculating attention values

torch has a function `scaled_dot_product_attention` to calculate the attention efficiently.

Lets first look at self-attention, what is the result?

In [ ]:
from torch.nn.functional import scaled_dot_product_attention

attention = scaled_dot_product_attention(query=v, value=v, key=v, dropout_p=0.0)
print(attention)
print(attention.shape)

The 10x4 result is actually the attention performed for all the queries.

### Attention with single query
We can also just calculate the result of a single query. So let's use different variables for the query and the key, but keep the values. We need as many keys as we have values. As indicated in the picture below, the result is then a weighted sum of the values

<img src="Attention.png" width="200">

In [ ]:
q = torch.rand([1,4])
k = torch.rand([10,4])
v = torch.rand([10,4])

attention = scaled_dot_product_attention(query=q, key=k, value=v)
print(attention)
print(attention.shape)

### Attention with different sizes for query and key

The query and key are used to calculate the attention weight, they do not need to be the same dimensions as the values. This is in fact used in the MultiHeadedAttention where the keys and queries are projected to smaller dimension.

In [ ]:
q = torch.rand([1,2])
k = torch.rand([10,2])
v = torch.rand([10,4])

attention = scaled_dot_product_attention(query=q, key=k, value=v)
print(attention)
print(attention.shape)

### Number of parameters

How many trainable variables are there in the attention calculation?

## Multiheaded attention

Let us now look at the multiheaded attention class in pytorch that can be used to build the transformer architecture.

We will first look at a single head. If we dont specify the embedding dimensions for the key and value, they will be the same as for the query.

In [ ]:
from torch.nn import MultiheadAttention
mha = MultiheadAttention(embed_dim=4, num_heads=1, kdim=None, vdim=None, batch_first=True)

In [ ]:
# unbatched example with only one query
q = torch.rand([1,4])
k = torch.rand([10,4])
v = torch.rand([10,4])
with torch.no_grad():
    result, weights = mha.forward(query=q, key=k, value=v)
print(result)
print(result.shape)
print(weights)
print(weights.shape)

## Multiheaded self attention
To calculate the self attention, we just set the query, key and value to the same vector

In [ ]:
# unbatched self attention
v = torch.rand([10,4])
with torch.no_grad():
    result, weights = mha.forward(query=v, key=v, value=v)
print(result)
print(result.shape)
print(weights)
print(weights.shape)

### Multiple heads

What happens to the result if we use multiple heads?

- The query and key values will get projected to lower dimension vectors of size (embedding / nr_heads) for each of the heads with a different project matrix.
- The output of a single head still has the dimension of the value embedding
- So if we concatenate the outputs of all heads the result will have a dimension of embedding * heads.
- The concatenated vector is therefor multiplied by another weight matrix to bring it back to the length of the value embedding.

The result sizes of multiheaded attention is therefor independent of the number of heads.

In [ ]:
# same with more heads, each head will project the q and k to a lower dimension, usually d/h
# so the embedding dimension must be devisable by num_heads
v = torch.rand([10,4])
mha = MultiheadAttention(embed_dim=4, num_heads=2, kdim=None, vdim=None, batch_first=True)
with torch.no_grad():
    result, weights = mha.forward(query=v, key=v, value=v)
print(result)
print(result.shape)
print(weights)
print(weights.shape)

## Trainable Parameters

How many trainable parameters does the multihead attention have, and what are they for?

In [ ]:
class OnlyAttention(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = MultiheadAttention(embed_dim=4, num_heads=1, kdim=4, vdim=4, batch_first=True)
    def forward(self, q, v, k):
        return self.attention.forward(query=q, key=k, value=v)

In [ ]:
mha = OnlyAttention()
summary(mha)
print(mha)

for name, param in mha.named_parameters():
    print(name)
    print(param)

## Transformer Encoder

Finally torch allows to build a transformer or the encoder and decoder seperately. For classification, as needed in the exercises, we only need a encoder.

In [ ]:
from torch.nn import TransformerEncoder, TransformerEncoderLayer

layer = TransformerEncoderLayer(d_model=4, nhead=1, dim_feedforward=16, dropout=0.1, batch_first=True)
encoder = TransformerEncoder(layer, num_layers=4)

v = torch.rand([10,4])
result_layer = layer.forward(v)
print(result_layer.shape)

result_encoder = encoder.forward(v)
print(result_encoder.shape)

In [ ]:
class TransformEnc(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = TransformerEncoderLayer(d_model=4, nhead=1, dim_feedforward=16, dropout=0.1, batch_first=True)
        self.encoder = TransformerEncoder(self.layer, num_layers=4)

    def forward(self, v):
        return encoder.forward(v)

In [ ]:
enc = TransformEnc()
summary(enc)

That was all for the tutorial about using attention and transfomers in torch.

## Exercise 1: Image Classification on CIFAR-10 with (own) Vis Transformer

We use the same data and setup as in the last exercise, but this time we would like to train visual transformer.


In [ ]:
transform = torchvision.transforms.Compose(
    [torchvision.transforms.ToTensor()])
data_train = torchvision.datasets.CIFAR10(root=data_dir, download=False, transform=transform)
data_test = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=False, transform=transform)

In [ ]:
print (f'train: {len(data_train)}')
print (f'train: {len(data_test)}')

In [ ]:
BATCH_SIZE = 64

data_train_loader = torch.utils.data.DataLoader(dataset=data_train, shuffle=True, batch_size=BATCH_SIZE)
data_test_loader = torch.utils.data.DataLoader(dataset=data_test, shuffle=False, batch_size=BATCH_SIZE)


In [ ]:
train_iter = iter(data_train_loader)
images, labels = next(train_iter)

plt.imshow(np.transpose(torchvision.utils.make_grid(images), (1, 2, 0)))

As outlined in the presentation, a transfomer needs a positional encoding. This is a standard implementation of the one from the original paper.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super(PositionalEncoding, self).__init__()

        # Create a matrix of shape (max_len, d_model) to store positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Calculate the positional encoding using sine and cosine functions
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register buffer so that it doesn't get updated during training
        self.pe = pe.unsqueeze(0)  # Shape becomes (1, max_len, d_model)
        
    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        seq_len = x.size(1)
        # Add positional encoding to the input embedding
        x = x + self.pe[:, :seq_len, :].to(x.device)
        return x

## Transformer Implementation

Now implement a transformer using the transformer encoding layers as before. However the first task is to split the input image into blocks. In the original paper, this was 16x16 blocks for the larger imagenet images. The cifar image are smaller (32x32), so we could use 8x8 blocks for example.

Then we use the encoded patches as input to the transformer.



In [ ]:
class VisTrans(nn.Module):
    def __init__(self,
                 embed_dim,
                 hidden_dim,
                 output_dim,
                 n_layers,
                 embedding_dropout,
                 n_heads,
                 dropout,
                 ):
        super(VisTrans, self).__init__()
        # YOUR CODE HERE
        raise NotImplementedError()
        

    def forward(self, x):

        # YOUR CODE HERE
        raise NotImplementedError()




In [ ]:
model = VisTrans(embed_dim=64, hidden_dim=64, output_dim=10, n_layers=2, embedding_dropout=0.0, n_heads=8, dropout=0.0)
print(model)
summary(model, input_size=(64, 3, 32,32))

In [ ]:
def get_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        # test if it worked
        x = torch.ones(1, device=device)
        print('Using CUDA device')

    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        x = torch.ones(1, device=device)
        print('Using MPS device')
    else:
        print('Using CPU')
        device = torch.device('cpu')
    return device

In [ ]:
device = get_device()

Training Loop

In [ ]:
def train(epochs: int, model, train_data, val_data, loss_function, optimizer, metrics, device):                                              
    input_count = 0
    step_count = 0
    model = model.to(device)
    
    for epoch in range(epochs):
        model.train()
        metrics.reset()
        for step, (inputs, labels) in enumerate(train_data):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Zero your gradients for every batch!
            optimizer.zero_grad()
            # calculate results
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            train_loss = loss_function(outputs, labels)
            train_loss.backward()
            optimizer.step()

            metrics.update(predicted, labels)
            train_acc = metrics.compute()
           
        model.eval()
        metrics.reset()
        val_loss = []
        val_steps = 0
        for step, (inputs, labels) in enumerate(val_data):
            inputs = inputs.to(device)
            labels = labels.to(device)
            with torch.no_grad():
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)

                val_loss.append(loss_function(outputs, labels).item())
                metrics.update(predicted, labels)

        val_acc = metrics.compute()
        val_loss_mean = np.mean(val_loss)

        print(f"Epoch {epoch:02} Train Loss: {train_loss:.3f}, Valid Loss: {val_loss_mean:.3f}, Train Accuracy: {train_acc:.2f} Valid Acc: {val_acc:.2f}")
    # return the last accuracy from the evaluation

                       

In [ ]:
num_epochs = 20
learning_rate = 0.0001
weight_decay = 0.0

# you can change the parameters to generate the model
# model = VisTrans(embed_dim=64, hidden_dim=64, output_dim=10, n_layers=2, embedding_dropout=0.0, n_heads=8, dropout=0.0
# YOUR CODE HERE
raise NotImplementedError()

my_metrics = MulticlassAccuracy(num_classes=10)

# initialize an optimizer and loss function and call train

my_optimizer = optim.Adam(model.parameters(), lr=learning_rate)
my_loss = nn.CrossEntropyLoss()
train(num_epochs, model,  data_train_loader, data_test_loader, my_loss, my_optimizer, my_metrics, device)


In [ ]:
# vision transformers do not work that well on small data sets :-)
# you should still be able to get about 40-50%
assert my_metrics.compute() > 0.4